# import

In [ ]:
import pandas as pd
import random
from utils.recbole_train_test import *
from utils.plot_utils import *
# from utils.model_utils import get_trainer

from datetime import datetime, timezone

from utils.generate_artificial_random_dataset import save_dataframe_2_atomic_file

# variables

In [ ]:
freq=6 # month
duration = 2*12//freq # 2 years split in xM buckets
n_parts = duration*2+1
d_keys = ['_pt'+str(i) for i in range(1, n_parts)]
# MODEL_VERSIONS = ['_pt1', '_pt2', '_pt3', '_pt4']
MODEL_VERSIONS = d_keys[:duration]


Ks = [1, 10, 20]
VM_K = Ks[2] # valid metric k, also used in heatmap matrix
VALID_METRIC = 'Recall@'+str(VM_K)
SEED = 2020
USE_GPU = False
SHOW_PROGRESS = False
SAVE_DATASET = False

# these are the default values
# TRAIN_NEG_SAMPLE_ARGS = {'distribution': 'uniform', 
#                          'sample_num': 1, 
#                          'alpha': 1.0, 
#                          'dynamic': False, 
#                          'candidate_num': 0}



SHUFFLE = False  # shuffle (bool): Whether or not to shuffle the training data before each epoch. Defaults to True.
EVAL_ARGS = {'split': {'LS': 'test_only'}, # leave-one-out sample type ['valid_and_test', 'valid_only', 'test_only']
                    'group_by': 'user',
                    'order': 'TO', # order (str): decides how we sort the data in .inter. random ordering or time ordering
                    'mode': 'uni100'}

METRICS = ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision', 'GiniIndex', 'TailPercentage']

# FILENAME_VERSION = '_ET_ND_LS.t_UD_SF_TO_UM.100'

data_types_dict = {'item_id':'object', # bc of drifted items, 'd_xxxxx'
                   'user_id':'object',
                   'timestamp':'float32'}

# functions

In [ ]:
def get_id2token(recbole_dataset, col):
    if col=='item_id':
        return recbole_dataset.id2token(recbole_dataset.iid_field, recbole_dataset.inter_feat.interaction[recbole_dataset.iid_field])
    elif col=='user_id':
        return recbole_dataset.id2token(recbole_dataset.uid_field, recbole_dataset.inter_feat.interaction[recbole_dataset.uid_field])
    return None


def recbole_ds_column_2_dataframe(recbole_dataset, col, data_type):
    recboleds_df = pd.DataFrame(get_id2token(recbole_dataset, col=col), columns=[col])
    recboleds_df[col] = recboleds_df[col].astype(data_type)
    return recboleds_df
        

def recbole_dataset_2_external_id_df(recbole_dataset, data_types_dict):
    i = recbole_ds_column_2_dataframe(recbole_dataset, col=recbole_dataset.iid_field, data_type=data_types_dict[recbole_dataset.iid_field])
    u = recbole_ds_column_2_dataframe(recbole_dataset, col=recbole_dataset.uid_field, data_type=data_types_dict[recbole_dataset.uid_field])

    df = pd.DataFrame({'user_id': u.user_id,
                       'item_id': i.item_id,
                       'timestamp': recbole_dataset.inter_feat.interaction[recbole_dataset.time_field].numpy()})
    df['timestamp'] = df['timestamp'].astype(data_types_dict['timestamp'])

    return df

# load pt4, save pt4's valid and test

In [ ]:
model_name = 'BPR' # for the sake of having one, BPR was chosen, but any other in theory yields the same results


save_path, base_filename, specs_str = ('processed_datasets/natural_data/goodreads/inter_dedup_coldstart_3stars_4x714k/',
                                        'more_2interQ_df', 
                                        'NPT_RD.50') 
BENCHMARK_FILENAMES = ['train', 'valid', 'test']

base_dataset_name = base_filename+'_'+specs_str


# from complete dataset (aka pt4)
part = MODEL_VERSIONS[-1]
dataset_name = base_dataset_name+part

parameter_dict = {  'dataset': dataset_name+'.inter',
                    'use_gpu':USE_GPU,
                    ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
                    'seed':SEED,
                    'state':'ERROR',
                    'data_path': save_path,
                    'save_dataset':SAVE_DATASET, #(bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
                    'checkpoint_dir':save_path+base_dataset_name,
                    'show_progress': SHOW_PROGRESS,
                    'shuffle': SHUFFLE,
                    ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
                    'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
                    # 'user_inter_num_interval':'[1,inf)',
                    # 'benchmark_filename': BENCHMARK_FILENAMES,
                    
                    ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
                    # 'train_neg_sample_args': TRAIN_NEG_SAMPLE_ARGS,
                    
                    ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
                    'eval_args': EVAL_ARGS,
                    'metrics': METRICS, 
                    'topk':Ks,
                    'valid_metric':VALID_METRIC          
                    }

# note: timestamps are not converted to internal ids, but need to be set to the correct dtype
_, _, internal_dataset_pt4,\
        internal_train_pt4,\
            internal_valid_pt4,\
                internal_test_pt4 = setup_config_and_dataset(model_name, dataset_name, parameter_dict)

external_valid_pt4 = recbole_dataset_2_external_id_df(internal_valid_pt4.dataset, data_types_dict)
save_dataframe_2_atomic_file(df=external_valid_pt4,
                             save_path=save_path,
                             base_filename=base_filename,
                             specs_str=specs_str+part,
                             benchmark_filename=BENCHMARK_FILENAMES[1])

external_test_pt4 = recbole_dataset_2_external_id_df(internal_test_pt4.dataset, data_types_dict)
save_dataframe_2_atomic_file(df=external_test_pt4,
                             save_path=save_path,
                             base_filename=base_filename,
                             specs_str=specs_str+part,
                             benchmark_filename=BENCHMARK_FILENAMES[-1])

# read Pre train 

In [ ]:
pretrain_dataset_name = base_filename+'_PT'
pretrain_dir = save_path+pretrain_dataset_name+'/'+pretrain_dataset_name+'.csv'
pretrain = pd.read_csv(pretrain_dir)

# remove test sets from pt4's train, save test sets, save cleaned train sets

In [ ]:
MODEL_VERSIONS[:-1]

In [ ]:
# remove all test sets from pt4's train set

parts_timestamps = {}
external_test_all_pts = pd.DataFrame()
for pti in MODEL_VERSIONS[:-1]: # all but pt4
    
    dataset_name = base_dataset_name+pti
    parameter_dict = {  'dataset': dataset_name+'.inter',
                        'use_gpu':USE_GPU,
                        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
                        'seed':SEED,
                        'state':'ERROR',
                        'data_path': save_path,
                        'save_dataset':SAVE_DATASET, #(bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
                        'checkpoint_dir':save_path+base_dataset_name,
                        'show_progress': SHOW_PROGRESS,
                        'shuffle': SHUFFLE,
                        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
                        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
                        # 'user_inter_num_interval':'[1,inf)',
                        # 'benchmark_filename': BENCHMARK_FILENAMES,
                        
                        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
                        # 'train_neg_sample_args': TRAIN_NEG_SAMPLE_ARGS,
                        
                        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
                        'eval_args': EVAL_ARGS,
                        'metrics': METRICS, 
                        'topk':Ks,
                        'valid_metric':VALID_METRIC          
                    }

    # note: timestamps are not converted to internal ids, but need to be set to the correct dtype
    _, _, internal_dataset_pti,\
            internal_train_pti,\
                internal_valid_pti,\
                    internal_test_pti = setup_config_and_dataset(model_name, dataset_name, parameter_dict)
    

    external_valid_pti = recbole_dataset_2_external_id_df(internal_valid_pti.dataset, data_types_dict)
    save_dataframe_2_atomic_file(df=external_valid_pti,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=specs_str+pti,
                                benchmark_filename=BENCHMARK_FILENAMES[1])

    external_test_pti = recbole_dataset_2_external_id_df(internal_test_pti.dataset, data_types_dict)
    save_dataframe_2_atomic_file(df=external_test_pti,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=specs_str+pti,
                                benchmark_filename=BENCHMARK_FILENAMES[-1])

    external_test_all_pts = pd.concat([external_test_all_pts, external_test_pti])



    ts = internal_train_pti.dataset.inter_feat.interaction[internal_train_pti.dataset.time_field].numpy().astype('float32')
    parts_timestamps[pti] = [ts.min(), ts.max()]


# remove from pt4's trainset and save it cleaned    
external_train_pt4 = recbole_dataset_2_external_id_df(internal_train_pt4.dataset, data_types_dict)

is_test_indicator_df = external_train_pt4.merge(external_test_all_pts, on=['user_id', 'item_id', 'timestamp'], how='left', indicator=True)
external_train_pt4_NoTest = is_test_indicator_df[is_test_indicator_df['_merge'] == 'left_only'].drop(columns=['_merge'])

# add pretrain to pt4's train set
ext_train_pt4_NT_PT = pd.concat([pretrain, external_train_pt4_NoTest])

save_dataframe_2_atomic_file(df=ext_train_pt4_NT_PT,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=specs_str+part,
                                benchmark_filename=BENCHMARK_FILENAMES[0])

# split pt4's cleaned-trainset in the different model parts and save it as the respective cleaned trainsets
for pti in MODEL_VERSIONS[:-1]:
    s, e = parts_timestamps[pti]
    external_train_pti_NoTest = external_train_pt4_NoTest.loc[(external_train_pt4_NoTest.timestamp>=s) & (external_train_pt4_NoTest.timestamp<=e),:]
    print('is there test interactions in both test and train sets?',pti, external_train_pti_NoTest.merge(external_test_all_pts, on=['user_id', 'item_id', 'timestamp'], how='left', indicator=True)._merge.value_counts())

    ext_train_pti_NT_PT = pd.concat([pretrain, external_train_pti_NoTest])

    save_dataframe_2_atomic_file(df=ext_train_pti_NT_PT,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=specs_str+pti,
                                benchmark_filename=BENCHMARK_FILENAMES[0])

